In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import pickle
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from config import Config
from OmniHealthFM import (
    RNA_Encoder,
    RNA_Decoder,
    OmniHealthFM_Model,
    CrossAttention,
    PatchShuffle
)

## Step 1: Model Pretraining

In [ ]:
config = Config() 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load gene embedding
with open("../model/Gene_Embedding.pkl", "rb") as f: 
    gene_embedding = pickle.load(f)
gene_embedding = gene_embedding.float()


# Load recount 3 data
gene_pretrain =  pd.read_csv("../data/demo_recount3_small.csv") 

In [ ]:
checkpoint_dir = '../model'
os.makedirs(checkpoint_dir, exist_ok=True)


class SingleModalDataset(Dataset):
    def __init__(self, rna_data):
        self.rna_data = rna_data

    def __len__(self):
        return len(self.rna_data)

    def __getitem__(self, idx):
        return {'rna': self.rna_data[idx]}


# Split into training/validation
train_rna, val_rna = train_test_split(gene_pretrain, test_size=0.3, random_state=123)
train_rna = torch.tensor(train_rna.values, dtype=torch.float32)
val_rna = torch.tensor(val_rna.values, dtype=torch.float32)

train_dataset = SingleModalDataset(train_rna)
val_dataset = SingleModalDataset(val_rna)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

# Initialize Model 
model = OmniHealthFM_Model(config, gene_embedding).to(config.device)


# Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr * config.batch_size / 64,
                              betas=(0.9, 0.999), weight_decay=1e-4)

lr_func = lambda epoch: min((epoch + 1) / (config.warmup_epoch + 1e-8),
                            0.5 * (math.cos(epoch / config.total_epoch * math.pi) + 1))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_func)

# Training 
early_stopping_patience = 5
best_val_loss = float('inf')
no_improvement_count = 0

for epoch in range(config.total_epoch):
    model.train()
    train_losses = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        rna = batch['rna'].to(config.device, non_blocking=True)
        recon_rna, mask, _,_,_ = model(rna)
        loss = torch.mean((recon_rna - rna) ** 2 * mask) / config.mask_ratio #recontruct masked genes
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_losses.append(loss.item())

    scheduler.step()
    torch.cuda.empty_cache()
    
    avg_train_loss = np.mean(train_losses)
    print(f"Epoch {epoch} - Train Loss: {avg_train_loss:.4f}")

    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            rna = batch['rna'].to(config.device, non_blocking=True)
            recon_rna, mask, _ ,_,_= model(rna)
            val_loss = torch.mean((recon_rna - rna) ** 2 * mask) / config.mask_ratio
            val_losses.append(val_loss.item())

    avg_val_loss = np.mean(val_losses)
    print(f"Epoch {epoch} - Val Loss: {avg_val_loss:.4f}")
    
   # Save model
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': avg_val_loss,
            'config': config.__dict__,
        }, os.path.join(checkpoint_dir, f'OmniHealthFM_Pretrained_Model_checkpoint_epoch{epoch}.pth'))

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        no_improvement_count = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
            'config': config.__dict__,
        }, os.path.join(checkpoint_dir, 'OmniHealthFM_Pretrained_Model.pth'))
    else:
        no_improvement_count += 1
        if no_improvement_count >= early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

Epoch 0 [Train]: 100%|██████████████████████| 7540/7540 [44:29<00:00,  2.82it/s]


Epoch 0 - Train Loss: 2.2457


Epoch 0 [Val]: 100%|████████████████████████| 3232/3232 [04:26<00:00, 12.14it/s]


Epoch 0 - Val Loss: 1.2727


Epoch 1 [Train]: 100%|██████████████████████| 7540/7540 [44:30<00:00,  2.82it/s]


...

Epoch 16 [Train]: 100%|█████████████████████| 7540/7540 [44:31<00:00,  2.82it/s]


Epoch 16 - Train Loss: 0.4551


Epoch 16 [Val]: 100%|███████████████████████| 3232/3232 [04:25<00:00, 12.17it/s]


Epoch 16 - Val Loss: 0.4420


Epoch 17 [Train]:  44%|█████████▏           | 3290/7540 [19:26<25:05,  2.82it/s]

## Step 2: Feature Extraction (Toy Example)

In [29]:
model = OmniHealthFM_Model(config, gene_embedding).to(config.device)

ckpt = torch.load(
    "../model/OmniHealthFM_Pretrained_Model.pth",
    map_location=config.device,
    weights_only=False
)

model.load_state_dict(ckpt["model_state_dict"], strict=False)

for p in model.parameters():
    p.requires_grad = False
model.eval()

@torch.no_grad()
def extract_features(X, batch_size=128):
    feats = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i + batch_size].to(device)
        recon, _, _, _, cls = model(batch)

        if isinstance(cls, list):
            cls = cls[-1]
            
        if cls.dim() == 3:
            cls = cls.squeeze(1)

        if recon.dim() == 3:
            recon = recon.squeeze(-1)

        z = torch.cat([cls, recon], dim=-1)
        feats.append(z.cpu().numpy())

    return np.vstack(feats)


In [33]:
gene_cli =  pd.read_csv("../data/toy_example.csv") 
gene=gene_cli.iloc[:, 4:]
X = torch.tensor(gene.values, dtype=torch.float32)
features = extract_features(X)
features[:5]

array([[-0.19645898, -0.14022431, -0.3928364 , ..., -0.49924353,
         0.01406043, -0.5003607 ],
       [-0.19029813, -0.18898144, -0.3869081 , ...,  0.04417167,
         1.8119365 ,  0.6045681 ],
       [-0.21027964, -0.13018474, -0.38913128, ..., -0.5005163 ,
         0.0119649 ,  0.61723864],
       [-0.19865605, -0.12295405, -0.3823913 , ...,  0.06936108,
         0.01125599, -0.5023491 ],
       [-0.21675003, -0.13732013, -0.41024807, ..., -0.49650285,
         0.01724144,  0.628055  ]], dtype=float32)